# finance / household / networth

Current net worth and the savings rate.

| Term | In This Notebook |
|---|---|
| **Intent** | Track the household's balance sheet and savings rate against the plan. Placeholder: twelve synthetic months |
| **Project Flavour** | `static`, local to the notebook (`model.md` 3.2) |
| **Data Splits** | `training` twelve synthetic months of net worth, income and spending, until the `bank` table exists; `golden` the exact request that becomes `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, target, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/household/networth"
MODEL_NAME = "finance_household_networth"
TARGETS = [{"kind": "local"}]
TRAINING = "household_networth_training_2026-11-05_2026-11-05"
GOLDEN = "household_networth_golden"
INPUT = "finance_household_networth_in"
OUTPUTS = ["finance_household_networth_current_dollars", "finance_household_networth_savings_rate_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic until the real series is loaded, so no query runs.

In [ ]:
training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {
        "month": list(range(12)),
        "net_worth_dollars": [1000000.0 + 5000.0 * month for month in range(12)],
        "income_dollars": [12000.0] * 12,
        "spending_dollars": [9000.0] * 12,
    },
)
print("training", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** take the latest net worth of `training` only (R6) and the savings rate as income less spending over income. Separating saving from market return needs the invested part, which the plan lists as data to get.

In [ ]:
income = float(training_pl["income_dollars"].sum())
spending = float(training_pl["spending_dollars"].sum())
CONSTANTS = dict(zip(OUTPUTS, [round(float(training_pl["net_worth_dollars"][-1]), 0), round((income - spending) / income * 100.0, 2)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [1055000.0, 25.0], "the summary did not give the latest net worth and the savings rate"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows, build the signature, write a `static` project whose constants are the fitted values, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, signature_input, signature_output
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "Unused input.")],
    [
        signature_output("finance_household_networth_current_dollars", "none", "Net worth at the latest month end, dollars.", datatype="FP64"),
        signature_output("finance_household_networth_savings_rate_pct", "percent", "Share of income not spent over the period, percent.", datatype="FP64"),
    ],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic monthly net worth, income and spending, training split",
        [training_path, golden_path],
        "latest net worth and the savings rate over the period",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")